In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes/notebooks
/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [2]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from megpypes.pipelines.meg_preprocessing import create_meg_preprocessing


/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [3]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# sessions
sessions = layout.get_sessions()
print(f"Sessions: {sessions}")
# task
layout.get_tasks()
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
print(f"Tasks: {layout.get_tasks()}")
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Sessions: []
Data types: ['meg']
[]
Tasks: ['MMNHCS', 'noise']


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [4]:
# Load configs
import os
import yaml
from nipype import config as nconfig
from nipype import logging

config_path = "config/config_effort.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

wf_config = config['workflow']
paths_config = config['paths']

# Configure Nipype logging (applies to all subprocesses)
logs_dir = Path(paths_config["workdir"]) / 'logs'
logs_dir.mkdir(parents=True, exist_ok=True)
config_dict = {
    'logging': {
        'log_directory': logs_dir,
        'log_to_file': True,
        'interface_level': 'DEBUG',
        'workflow_level': 'DEBUG',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
}
nconfig.update_config(config_dict)
logging.update_logging(nconfig)

# Create workflow
wf = create_meg_preprocessing(
    basedir=paths_config['basedir'], 
    workdir=paths_config['workdir'], 
    output_dir=paths_config['outputdir'],
    file_templates=paths_config.get('file_templates', None),
    iterable_fields=paths_config.get('iterable_fields', None),
    iterable_values=paths_config.get('iterable_values', None),
    pipeline_config=config['pipeline_config']
    )

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow

n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/effort
Iterating over fields: ['subject', 'session']
Provided iterable values: {'subject': ['0001'], 'session': ['01', '02']}
Processing iterable field: subject
Using custom values for field: subject
Processing iterable field: session
Using custom values for field: session
Final iterables dict: {'subject': ['0001'], 'session': ['01', '02']}
260324-13:15:46,997 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): No edge data
260324-13:15:46,997 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): new edge data: {'connect': [('subject', 'subject')]}
260324-13:15:46,997 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): Edge data exists: {'connect': [('subject', 'subject')]}
260324-13:15:46,998 nipype.workflow DEBUG:
	 (megpreproc.infosource, megpreproc.selectfiles): new edge data: {'connect': [('subject', 'subject'), ('session', 'session')]}
Valid inputs for initial_p

/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/nipype/external/cloghandler.py:145: UserWarning: The given 'filename' should be an absolute path.  If your application calls os.chdir(), your logs may get messed up. Use 'supress_abs_warn=True' to hide this message.
  warn(


260324-13:15:47,212 nipype.workflow INFO:
	 Generated workflow graph: workdir/megpreproc/graph.png (graph2use=colored, simple_form=True).
Workflow graph saved to: workdir/megpreproc/graph.png
Running with 8 workers
260324-13:15:47,215 nipype.workflow DEBUG:
	 Creating flat graph for workflow: megpreproc
260324-13:15:47,216 nipype.workflow DEBUG:
	 expanding workflow: megpreproc
260324-13:15:47,217 nipype.workflow DEBUG:
	 processing node: megpreproc.infosource
260324-13:15:47,217 nipype.workflow DEBUG:
	 processing node: megpreproc.selectfiles
260324-13:15:47,217 nipype.workflow DEBUG:
	 processing node: megpreproc.initial_preproc
260324-13:15:47,217 nipype.workflow DEBUG:
	 processing node: megpreproc.artifact_rejection
260324-13:15:47,217 nipype.workflow DEBUG:
	 processing node: megpreproc.datasink
260324-13:15:47,218 nipype.workflow DEBUG:
	 processing node: megpreproc.epoching
260324-13:15:47,218 nipype.workflow DEBUG:
	 processing node: megpreproc.collect_epochs
260324-13:15:47,2